# Gemfinder — October 9 Backup Methods

**Goal:** implement and evaluate a Monte Carlo empirical-quantile fallback for candidate Gamma failures identified in the September 25 analysis.

**Model warning:** The September 25 Python simulator uses observed response counts as *proxies* for latent types. That is not a validated reproduction of the original R simulator. Treat its results as exploratory until confirmed.

## 1. Check Python environment

In [ ]:
import sys
from pathlib import Path
print(sys.version)
print("Working directory:", Path.cwd())
# Uncomment in JupyterLab if dependencies are missing:
# %pip install -r requirements.txt

## 2. Run the self-contained demo
This synthetic demo requires no original survey data and checks that the code works.

In [ ]:
import subprocess
subprocess.run([sys.executable, "evaluate_backup_methods.py", "--demo"], check=True)

In [ ]:
import pandas as pd
from IPython.display import display
results = pd.read_csv("results/backup_method_comparison.csv")
display(results[["instance_id", "method", "q95_estimate", "reference_q95", "q95_relative_error", "tail_absolute_error", "source_status"]])

## 3. Run on September 25 candidate cases (exploratory only)
Set the paths to the actual location of your September 25 folder. The case CSV needs `instance_id`, `question`, `x_counts`, and `y_counts`. The P matrix can be `P.RData` or a long-form CSV. **No original survey CSV is required for this mode.**

In [ ]:
from pathlib import Path
CASE_FILE = Path("../Gamma_Fit_Evaluation_and_Failure_Cases/results/worst_10_gamma_cases.csv")
P_FILE = Path("../Gamma_Fit_Evaluation_and_Failure_Cases/P.RData")
print("Case CSV exists:", CASE_FILE.exists())
print("P matrix exists:", P_FILE.exists())

In [ ]:
if CASE_FILE.exists() and P_FILE.exists():
    subprocess.run([sys.executable, "evaluate_backup_methods.py",
                    "--case-file", str(CASE_FILE), "--p-file", str(P_FILE),
                    "--n-cases", "3", "--output-dir", "results/925_proxy_comparison"], check=True)
    display(pd.read_csv("results/925_proxy_comparison/backup_method_comparison.csv"))
else:
    print("Add your 9.25 CSV and P file, then rerun this cell.")

## 4. Preferred: validated original-model simulation draws
Place one CSV per case in `validated_draws/`, each with a column named `statistic` and at least 14,000 draws (for defaults). These should come from the **original R latent-type model**, not from the response-as-latent-type proxy.

In [ ]:
DRAW_DIR = Path("validated_draws")
if DRAW_DIR.exists() and list(DRAW_DIR.glob("*.csv")):
    subprocess.run([sys.executable, "evaluate_backup_methods.py",
                    "--draws-dir", str(DRAW_DIR), "--output-dir", "results/validated_model_comparison"], check=True)
    display(pd.read_csv("results/validated_model_comparison/backup_method_comparison.csv"))
else:
    print("No validated model draws yet. Ask the team for output from the original R simulator.")

## 5. October 9 deliverables

- Working Monte Carlo empirical quantile/tail estimator (`backup_methods.py`).
- Independent-sample comparison with Gamma (`evaluate_backup_methods.py`).
- CSV and figures in `results/`.
- Clearly labeled exploratory/validated provenance.

**For October 16:** repeat seeds, benchmark full simulation runtime, verify genuine failure cases, compare other backups, and recommend a deployment rule.